In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [4]:
from grb.io import read_data, filter_data
from grb.extinction import correct_host_galaxy_extinction

In [5]:
df = read_data("circular", correct_galactic_extinction=True, add_converted_flux=True)
filtered_df = filter_data(df, filter_name="Ic")
corrected_df = correct_host_galaxy_extinction(filtered_df, 0.126)

In [ ]:
from grb.modeling import add_observation
import grb.vegas_compat  # thread-safety patch for VegasAfterglow's bilby samplers (npool > 1)
from VegasAfterglow import ParamDef, Scale, Fitter
from grb.const import D_L, REDSHIFT


In [7]:
df_xrt = read_data("xrt")

In [ ]:
fitter = Fitter(
    z=REDSHIFT,           # Redshift
    lumi_dist=D_L,        # Luminosity distance [cm]
    medium="wind",        # Ambient medium type
    jet="tophat",         # Jet structure type
    rvs_shock=True,       # Include reverse shock
    fwd_ssc=True,         # Forward shock inverse Compton
    rvs_ssc=False,        # Reverse shock inverse Compton
    kn=True,              # Klein-Nishina corrections
    magnetar=False,       # Magnetar energy injection
    rtol=1e-5,            # Numerical tolerance
)
# (ssc_cooling option no longer exists in VegasAfterglow 2.x)

add_observation(corrected_df, fitter, input_type="flux_density", label="Ic")
add_observation(df_xrt, fitter, input_type="flux", label="XRT")


In [ ]:
# Basic parameter set
params = [
    ParamDef("E_iso",   1e50,  1e54,  Scale.LOG),     # Isotropic energy in erg
    ParamDef("Gamma0",    10,   500,  Scale.LOG),     # Lorentz factor
    ParamDef("theta_c", 0.01,   0.5,  Scale.LINEAR),  # Opening angle in radians
    ParamDef("theta_v",    0,     0,  Scale.FIXED),   # Viewing angle (on-axis) in radians
    ParamDef("A_star",  1e-3,   1.0,  Scale.LOG), 
    ParamDef("p",        2.1,   2.8,  Scale.LINEAR),  # Electron spectral index
    ParamDef("eps_e",   1e-3,   0.5,  Scale.LOG),     # Electron energy fraction
    ParamDef("eps_B",   1e-5,   0.1,  Scale.LOG),     # Magnetic energy fraction
    ParamDef("xi_e",     0.1,   1.0,  Scale.LINEAR),  # Fraction of accelerated electrons
    ParamDef("tau",        1,   100,  Scale.LOG),
        # Forward + reverse microphysics
    ParamDef("p_r",      2.1,   2.8,  Scale.LINEAR),
    ParamDef("eps_e_r", 1e-3,   0.5,  Scale.LOG),
    ParamDef("eps_B_r", 1e-5,   0.1,  Scale.LOG),
    ParamDef("xi_e_r",   0.1,   1.0,  Scale.LINEAR),
]


In [ ]:
result = fitter.fit(
    params,
    resolution=(0.15, 0.5, 10),     # Grid resolution (phi, theta, t)
    sampler="dynesty",             # Nested sampling algorithm
    nlive=1000,                    # Number of live points
    walks=100,                     # Number of random walks per live point
    dlogz=0.5,                     # Stopping criterion (evidence tolerance)
    npool=8,                       # Number of parallel processes
    top_k=10,                      # Number of best-fit parameters to return
)
